In [1]:
import sqlite3

DB_PATH = "capability_ontology.db"

conn = sqlite3.connect(DB_PATH)

try:
    # Get all pending candidate IDs first
    pending_ids = conn.execute(
        "SELECT candidate_id FROM candidate_patterns "
        "WHERE status = 'PENDING'"
    ).fetchall()

    # Delete their evidence records
    for (candidate_id,) in pending_ids:
        conn.execute(
            "DELETE FROM candidate_pattern_evidence "
            "WHERE candidate_id = ?",
            (candidate_id,)
        )

    # Delete the pending candidates
    cursor = conn.execute(
        "DELETE FROM candidate_patterns "
        "WHERE status = 'PENDING'"
    )

    conn.commit()

    print(f"Deleted {cursor.rowcount} pending candidate(s).")

finally:
    conn.close()

Deleted 6 pending candidate(s).


In [2]:
import sqlite3
from pathlib import Path

DB_PATH = Path("capability_ontology.db")

conn = sqlite3.connect(DB_PATH)

try:
    # ------------------------------------------------------------
    # 1. Remove Module 10 candidate review data
    # ------------------------------------------------------------

    conn.execute("""
        DELETE FROM candidate_pattern_evidence
    """)

    conn.execute("""
        DELETE FROM candidate_patterns
    """)

    # ------------------------------------------------------------
    # 2. Remove MITRE ATLAS data acquired by Module 10
    # ------------------------------------------------------------

    conn.execute("""
        DELETE FROM source_techniques
        WHERE source_id LIKE 'MITRE-ATLAS-%'
    """)

    conn.execute("""
        DELETE FROM threat_sources
        WHERE source_id LIKE 'MITRE-ATLAS-%'
    """)

    # ------------------------------------------------------------
    # 3. Remove LLM-generated attack patterns.
    #
    # Your original patterns are P1-P9.
    # Module 10 creates P10, P11, P12, etc.
    #
    # Therefore keep P1-P9 and delete P10+.
    # ------------------------------------------------------------

    rows = conn.execute("""
        SELECT pattern_id
        FROM attack_patterns
    """).fetchall()

    deleted_patterns = 0

    for (pattern_id,) in rows:
        try:
            number = int(str(pattern_id)[1:])

            if number >= 10:
                conn.execute(
                    """
                    DELETE FROM attack_patterns
                    WHERE pattern_id = ?
                    """,
                    (pattern_id,)
                )
                deleted_patterns += 1

        except (ValueError, IndexError):
            # Ignore non-standard pattern IDs
            pass

    conn.commit()

    print("=" * 60)
    print("DATABASE RESET COMPLETE")
    print("=" * 60)
    print(f"LLM-generated attack patterns removed : {deleted_patterns}")
    print("Module 10 candidate queue             : CLEARED")
    print("MITRE ATLAS source data               : CLEARED")
    print("Original P1-P9 patterns               : PRESERVED")
    print("=" * 60)

finally:
    conn.close()

DATABASE RESET COMPLETE
LLM-generated attack patterns removed : 0
Module 10 candidate queue             : CLEARED
MITRE ATLAS source data               : CLEARED
Original P1-P9 patterns               : PRESERVED
